# DiffusionSat speedup experimentation

The goal of this notebook is to improve the DiffusionSat backbone inference times.

Resources:
- https://huggingface.co/docs/diffusers/v0.26.1/tutorials/fast_diffusion
- https://github.com/chengzeyi/stable-fast
- https://dev.to/codesmart_1/improve-your-inference-speed-for-stable-diffusion-pipelines-2h4

### Experiments

Initial config:
- NV A40 GPU
- 128 samples, bs 32, num_workers 4, max_ts 50, save_ts [48, 56, 42], layer_idxs down attn1 all, low_cpu_mem_usage=False
- no explicit pytorch optims

Baseline: 00:36<00:00,  9.22s/it, ~6973MiB

+cast everything to torch.bfloat16: 00:36<00:00,  9.14s/it, ~5185MiB / won't do

+remove torch.cuda.empty_cache() from LDMExtractor: 00:30<00:00,  7.65s/it, ~14597MiB / keeping this

+torch.set_float32_matmul_precision("high"): 00:30<00:00,  7.73s/it, 14597MiB / keeping this as no harm

+remove GPU/CPU ping-pong from LDMExtractor: 00:31<00:00,  7.78s/it, 14619MiB / no impact, keeping as no harm

+SDPA for UNet unet.set_attn_processor(AttnProcessor2_0()): 00:31<00:00,  7.83s/it, 14619MiB / no impact, keeping as no harm

+cast all to torch.bfloat16: 00:29<00:00,  7.48s/it, 7557MiB / keeping as no harm

+torch.compile(self.unet, mode="reduce-overhead") on UNet: didn't run since we're using hooks which are incompat with compile / won't use

+offload CLIP encoder after init: 00:29<00:00,  7.43s/it, 4.3samples/s, 6021MiB / good reduction in VRAM

+bs=64: 00:28<00:00, 14.17s/it, 4.5samples/s, 9125MiB / no significant degradation or speedup

+bs=128, 512 samples: 01:45<00:00, 26.41s/it, 4.84samples/s, 15357MiB / moderate speedup over 4.3samples/s

+one captured timestep: 00:26<00:00, 26.81s/it / no changes, won't do

+max_timesteps=10, save_timesteps=[9,8,7], 256 samples: 00:13<00:00,  6.63s/it, 19.3samples/s / speedup is significant; not sure about quality, though

In [1]:
%load_ext autoreload
%autoreload 2

import os
import warnings
from pathlib import Path

import torch
from tqdm import tqdm

# Filter warnings.
warnings.filterwarnings("ignore", message=".*invalid escape sequence.*")


VISLOC_ROOT = Path(os.environ["VISLOC_ROOT"])
DIFFUSIONSAT_256_CHCKPT = Path(os.environ["DIFFUSIONSAT_256_CHCKPT"])

RANDOM_SEED = 42
DEVICE = torch.device("cuda")
NUM_WORKERS = 4

In [ ]:
from src.backbone import DiffusionSatBackbone
from src.ldm_extractor import LDMExtractorCfg

# Torch performance.
torch.set_float32_matmul_precision("high")

# Initialize the backbone once to avoid re-loading unet to VRAM each time.
diffusionsat_backbone = DiffusionSatBackbone(DIFFUSIONSAT_256_CHCKPT, DEVICE, dtype=torch.bfloat16)

ldm_extractor_cfg = LDMExtractorCfg(
  batch_size=128,
  save_timesteps=[48, 46, 42],
  num_timesteps=50,
  layer_idxs={"down_blocks": {"attn1": "all"}},
)
diffusionsat_backbone.set_ldm_extractor_cfg(ldm_extractor_cfg)

/root/diffusion-vpr/.venv/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/diffusion-vpr/.venv/lib/python3.10/site-packages/diffusers/models/cross_attention.py:30: FutureWarning: Importing from cross_attention is deprecated. Please import from diffusers.models.attention_processor instead.
  deprecate(
Some weights of the model checkpoint at /workspace/checkpoints/finetune_sd21_256_sn-satlas-fmow_snr5_md7norm_bs64_trimmed were not used when initializing SatUNet: ['metadata_embedding.6.linear_1.weight', 'metadata_embedding.2.linear_1.weight', 'metadata_embedding.1.linear_2.weight', 'metadata_embedding.6.linear_1.bias', 'metadata_embedding.5.linear_2.bias', 'metadata_embedding.4.linear_1.bias', 'metadata_embe

In [3]:
from torch.utils.data import Subset

from src.data_transforms import inference_transforms
from src.datasets import SatChunkDataset

gallery_dataset = SatChunkDataset(
  root=VISLOC_ROOT,
  flight_id="03",
  transform=inference_transforms,
)
gallery_dataset = Subset(gallery_dataset, range(256))
loader = torch.utils.data.DataLoader(
  gallery_dataset,
  batch_size=ldm_extractor_cfg.batch_size,
  shuffle=False,
  num_workers=NUM_WORKERS,
  pin_memory=True,
)


In [4]:
for imgs, _, _ in tqdm(loader, desc="Extracting features"):
  imgs = imgs.to(DEVICE, dtype=torch.bfloat16)
  features = diffusionsat_backbone(imgs)

Extracting features:   0%|          | 0/2 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 2/2 [00:13<00:00,  6.63s/it]
